# Depth Regularization in 3D Gaussian Splatting

## 1. 개요 (Overview)

Depth regularization은 각 입력 이미지의 depth map을 prior로 사용하여 학습을 보조하는 기능입니다. 원본 README에서는 untextured 영역(예: 도로)에서 특히 도움이 될 수 있고 floaters를 줄일 수 있다고 설명하며, 장면에 따라 개선 폭이 작거나 오히려 나빠질 수도 있다고 언급합니다.

**핵심 아이디어:**
- Monocular depth network (Depth Anything V2)로 각 이미지의 상대적 깊이 맵 예측
- COLMAP sparse points를 사용하여 상대 깊이를 절대 스케일로 정렬
- 학습 중 렌더링된 깊이와 예측된 깊이 사이의 L1 loss 최소화
- Gaussians가 올바른 3D 위치로 수렴하도록 유도

---

## 2. Depth Regularization 파이프라인

```
┌─────────────────────────────────────────────────────────────────┐
│                    Depth Regularization 전체 흐름                │
└─────────────────────────────────────────────────────────────────┘

[Step 1] Monocular Depth Prediction
         │
         ├─→ Depth Anything V2 실행
         │   └─→ 각 이미지의 상대 깊이 맵 생성 (16-bit PNG)
         │
[Step 2] Depth Scale Alignment (make_depth_scale.py)
         │
         ├─→ COLMAP sparse points를 각 이미지에 투영
         ├─→ Monocular depth와 COLMAP depth 비교
         └─→ 이미지별 scale/offset 계산 → depth_params.json 생성
         │
[Step 3] Training with Depth Loss
         │
         ├─→ 각 iteration마다 depth 렌더링
         ├─→ scale/offset으로 monocular depth 정렬
         └─→ L1 depth loss 계산 및 역전파
```



---

## 3. Step 1: Monocular Depth Prediction

### 3.1 Depth Anything V2 개요

3DGS README는 real-world 데이터셋에서 depth regularization을 쓰기 위해, 각 입력 이미지에 대해 Depth Anything v2로 depth map을 생성하는 절차를 제시합니다.

Depth Anything V2는 **affine-invariant inverse depth**를 생성합니다:
- **Affine-invariant**: scale과 shift에 불변 (즉, $d' = a \cdot d + b$ 형태의 변환으로는 절대 스케일 결정 불가)
- **Inverse depth**: $1/\text{depth}$ 형태로 출력 (가까운 물체에서 높은 정밀도, 먼 물체/무한대는 0으로 자연스럽게 표현)
- 따라서 COLMAP의 metric depth와 정렬하기 위해 scale/offset을 구해야 함

---

## 4. Step 2: Depth Scale Alignment

### 4.1 문제 정의

Monocular depth는 **상대적 깊이만** 제공하므로, COLMAP의 절대 스케일과 정렬 필요:

```
COLMAP depth:       [0.5, 1.2, 3.8, 7.5] meters
Monocular depth:    [0.82, 0.65, 0.31, 0.18] (상대값)
```



**목표:** 각 이미지에 대해 `scale`, `offset` 계산:

```
aligned_mono_depth = mono_depth * scale + offset
```



### 4.2 데이터 매칭 방법 ($D_{col}$과 $D_{mono}$의 생성)

두 깊이 값을 비교하기 위해서는, **정확히 같은 이미지 픽셀 위치 $(u, v)$** 에서의 값을 가져와야 합니다. 이를 위해 COLMAP의 3D 포인트를 2D 이미지 평면에 투영하는 과정을 거칩니다.

**1. 유효한 COLMAP 3D 점 찾기**
특정 이미지 $I$가 관측하고 있는 3D 점들의 집합 $P_{3D}$를 가져옵니다. COLMAP의 `images.bin`에는 각 이미지가 어떤 3D 점(`point3D_ids`)을 보고 있는지 기록되어 있습니다.

**2. 3D → 2D 투영 (Projection)**
3D 월드 좌표 $\mathbf{X}_w$를 카메라 내부/외부 파라미터를 사용하여 픽셀 좌표 $(u, v)$로 변환합니다.

$$
\mathbf{X}_{cam} = \mathbf{R} \cdot \mathbf{X}_w + \mathbf{t}
$$
$$
u = f_x \frac{X_{cam}}{Z_{cam}} + c_x, \quad v = f_y \frac{Y_{cam}}{Z_{cam}} + c_y
$$

여기서 $Z_{cam}$이 3D 상의 실제 깊이(Metric Depth)입니다.
- $depth_{col} = Z_{cam}$
- $inv\_depth_{col} = 1 / Z_{cam}$

**3. Monocular Depth 샘플링**
위에서 계산된 픽셀 좌표 $(u, v)$ 위치의 Monocular Depth 맵 값을 읽어옵니다. 좌표가 정수가 아닐 경우 Bilinear Interpolation(쌍선형 보간)을 사용합니다.
- $inv\_depth_{mono} = \text{DepthMap}(u, v)$

**4. 데이터 쌍 구성**
이 과정을 모든 유효한 3D 점에 대해 반복하여 대응 쌍 리스트를 만듭니다.
$$
D_{col} = \{ d_{c1}, d_{c2}, \dots, d_{cN} \}
$$
$$
D_{mono} = \{ d_{m1}, d_{m2}, \dots, d_{mN} \}
$$

### 4.3 make_depth_scale.py 코드 분석

#### 전체 구조

In [ ]:
import numpy as np
import argparse
import cv2
from joblib import delayed, Parallel
import json
from read_write_model import *

def get_scales(key, cameras, images, points3d_ordered, args):
    image_meta = images[key]
    cam_intrinsic = cameras[image_meta.camera_id]

    pts_idx = images_metas[key].point3D_ids

    mask = pts_idx >= 0
    mask *= pts_idx < len(points3d_ordered)

    pts_idx = pts_idx[mask]
    valid_xys = image_meta.xys[mask]

    if len(pts_idx) > 0:
        pts = points3d_ordered[pts_idx]
    else:
        pts = np.array([0, 0, 0])

    R = qvec2rotmat(image_meta.qvec)
    pts = np.dot(pts, R.T) + image_meta.tvec

    invcolmapdepth = 1. / pts[..., 2] 
    n_remove = len(image_meta.name.split('.')[-1]) + 1
    invmonodepthmap = cv2.imread(f"{args.depths_dir}/{image_meta.name[:-n_remove]}.png", cv2.IMREAD_UNCHANGED)
    
    if invmonodepthmap is None:
        return None
    
    if invmonodepthmap.ndim != 2:
        invmonodepthmap = invmonodepthmap[..., 0]

    invmonodepthmap = invmonodepthmap.astype(np.float32) / (2**16)
    s = invmonodepthmap.shape[0] / cam_intrinsic.height

    maps = (valid_xys * s).astype(np.float32)
    valid = (
        (maps[..., 0] >= 0) * 
        (maps[..., 1] >= 0) * 
        (maps[..., 0] < cam_intrinsic.width * s) * 
        (maps[..., 1] < cam_intrinsic.height * s) * (invcolmapdepth > 0))
    
    if valid.sum() > 10 and (invcolmapdepth.max() - invcolmapdepth.min()) > 1e-3:
        maps = maps[valid, :]
        invcolmapdepth = invcolmapdepth[valid]
        invmonodepth = cv2.remap(invmonodepthmap, maps[..., 0], maps[..., 1], interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)[..., 0]
        
        ## Median / dev
        t_colmap = np.median(invcolmapdepth)
        s_colmap = np.mean(np.abs(invcolmapdepth - t_colmap))

        t_mono = np.median(invmonodepth)
        s_mono = np.mean(np.abs(invmonodepth - t_mono))
        scale = s_colmap / s_mono
        offset = t_colmap - t_mono * scale
    else:
        scale = 0
        offset = 0
    return {"image_name": image_meta.name[:-n_remove], "scale": scale, "offset": offset}



#### 핵심 알고리즘: MAD (Median Absolute Deviation) Alignment

**왜 Median/MAD인가?**
- **Robustness**: COLMAP 포인트 클라우드는 outlier(잘못 매칭된 점)가 포함될 수 있습니다. 최소자승법(Least Squares)은 outlier에 의해 scale이 크게 왜곡되지만, Median은 영향을 거의 받지 않습니다.
- **기법의 유래**: 이 방식은 Monocular Depth Estimation 평가나 MVS 통합 관련 연구들에서 정립된 방식입니다.

### 4.4 출력: depth_params.json

In [ ]:
{
  "IMG_0001": {
    "scale": 0.8542,
    "offset": 0.1234
  },
  "IMG_0002": {
    "scale": 0.0,
    "offset": 0.0
  }
}



**`scale == 0`인 경우:**
- 유효한 sparse points가 10개 미만
- 또는 depth 범위가 너무 작음 (`max - min < 1e-3`)
- → 해당 이미지는 depth regularization 사용 안 함

**추가 정보: `med_scale` 계산**

`dataset_readers.py`에서 모든 이미지의 median scale 계산:

In [ ]:
# dataset_readers.py (line 157-177)
depth_params_file = os.path.join(path, "sparse/0", "depth_params.json")

depths_params = None
if depths != "":
  with open(depth_params_file, "r") as f:
    depths_params = json.load(f)
  all_scales = np.array([depths_params[key]["scale"] for key in depths_params])
  if (all_scales > 0).sum():
    med_scale = np.median(all_scales[all_scales > 0])
  else:
    med_scale = 0
  for key in depths_params:
    depths_params[key]["med_scale"] = med_scale



---

## 5. Step 3: Training with Depth Loss

### 5.1 Camera 클래스에서 Depth 로드 (코드 위치: [scene/cameras.py](../../references/gaussian-splatting/scene/cameras.py))

In [ ]:
class Camera(nn.Module):
    # ...
    # [scene/cameras.py]의 60-79번 줄

        self.invdepthmap = None
        self.depth_reliable = False
        if invdepthmap is not None:
            self.depth_mask = torch.ones_like(self.alpha_mask)
            self.invdepthmap = cv2.resize(invdepthmap, resolution)
            self.invdepthmap[self.invdepthmap < 0] = 0
            self.depth_reliable = True

            if depth_params is not None:
                if depth_params["scale"] < 0.2 * depth_params["med_scale"] or depth_params["scale"] > 5 * depth_params["med_scale"]:
                    self.depth_reliable = False
                    self.depth_mask *= 0
                
                if depth_params["scale"] > 0:
                    self.invdepthmap = self.invdepthmap * depth_params["scale"] + depth_params["offset"]

            if self.invdepthmap.ndim != 2:
                self.invdepthmap = self.invdepthmap[..., 0]
            self.invdepthmap = torch.from_numpy(self.invdepthmap[None]).to(self.data_device)



**핵심 포인트:**
1. **`depth_reliable` 플래그:**
   - `True`: Depth loss 계산에 사용
   - `False`: 해당 이미지는 depth regularization 스킵
   
2. **Outlier 필터링:**
   - Scale이 `0.2 * med_scale ~ 5 * med_scale` 범위 밖 → 신뢰 불가
   - 예: `med_scale=1.0`일 때, `scale=0.1` 또는 `scale=6.0`이면 제외

3. **`depth_mask`:**
   - 기본값: 1 (모든 픽셀 사용)
   - Unreliable depth면 0으로 설정 → loss 계산에서 제외

### 5.2 Training Loop에서 Depth Loss 계산 (코드 위치: [train.py](../../references/gaussian-splatting/train.py))

[train.py](../../references/gaussian-splatting/train.py)의 128-141번 줄 `depth regularization` 블록:

In [ ]:
# train.py

        # Depth regularization
        Ll1depth_pure = 0.0
        if depth_l1_weight(iteration) > 0 and viewpoint_cam.depth_reliable:
            invDepth = render_pkg["depth"]
            mono_invdepth = viewpoint_cam.invdepthmap.cuda()
            depth_mask = viewpoint_cam.depth_mask.cuda()

            Ll1depth_pure = torch.abs((invDepth  - mono_invdepth) * depth_mask).mean()
            Ll1depth = depth_l1_weight(iteration) * Ll1depth_pure 
            loss += Ll1depth
            Ll1depth = Ll1depth.item()
        else:
            Ll1depth = 0



**참고 (원본 구현 기준):** depth regularization은 inverse depth 맵을 직접 비교합니다. (`train.py`의 `invDepth = render_pkg["depth"]` 및 `viewpoint_cam.invdepthmap` 참조)

---

### 5.3 Depth 가중치 스케줄

Depth loss의 가중치는 exponential decay 함수를 사용하여 학습 초기에는 강하게, 후반으로 갈수록 약하게 적용됩니다:

In [ ]:
# train.py (line 64)
depth_l1_weight = get_expon_lr_func(opt.depth_l1_weight_init, opt.depth_l1_weight_final, max_steps=opt.iterations)



**파라미터 (arguments/__init__.py, line 92-93):**

In [ ]:
self.depth_l1_weight_init = 1.0    # 초기 가중치
self.depth_l1_weight_final = 0.01  # 최종 가중치



**Exponential Decay 함수 (utils/general_utils.py):**
원본 구현의 `get_expon_lr_func(...)`를 그대로 사용합니다. (코드 위치: [utils/general_utils.py](../../references/gaussian-splatting/utils/general_utils.py))

---

## 6. 참고 자료 (References)

**논문:**
- [3D Gaussian Splatting for Real-Time Radiance Field Rendering](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/) (Kerbl et al., SIGGRAPH 2023)
- [A Hierarchical 3D Gaussian Representation for Real-Time Rendering of Very Large Datasets](https://repo-sam.inria.fr/fungraph/hierarchical-3d-gaussians/) (Kerbl et al., SIGGRAPH 2024) - Depth regularization 방법이 원본 3DGS에 통합됨
- [Towards Robust Monocular Depth Estimation: Mixing Datasets for Zero-shot Cross-dataset Transfer](https://arxiv.org/abs/1907.01341) (Ranftl et al., TPAMI 2022) - Inverse depth 기반 scale/offset 정렬 방법

**코드:**
- [gaussian-splatting](https://github.com/graphdeco-inria/gaussian-splatting)
- [Depth-Anything-V2](https://github.com/DepthAnything/Depth-Anything-V2)
- [COLMAP](https://github.com/colmap/colmap)